before starting genie code - select desired compute in top right, I use serverless v5 if allowed, otherwise use your own personal all purpose compute cluster, also see where catalog and schema is mentioned below and update to your catalog/schema

## the objective of this notebook is to read data from zip files stored in a volume into appropriate delta tables to do downstream analysis on the Seattle market

Ingest Steps:
- read data in from zip files in raw_data schema and create appropriate delta tables
- Use cmegdemos_catalog.network_enablement_demo_test (parameterize these values throughout the notebook)
- raw data zip files are in the raw_data volume in this schema
- Load FCC BDC H3 hexagon coverage data - filter to only Seattle area
- Importing Microsoft building footprints as polygon geometries using Databricks Spatial SQL and save to delta table 
- load in 310.csv.gz file from Open Cell ID for tower data in seatle market but filter to only t mobile
- make sure tables are ready for downstream analysis with H3 and ST functions

prompt to genie code: see instructions in this notebook and get to work, feel free to add an additional markdown below with refined instructions after you create your plan and/or ask any questions to me before starting on the work

## Refined Ingest Plan

**Catalog:** `cmegdemos_catalog` | **Schema:** `network_enablement_demo_test` | **Volume:** `raw_data`

**Source files in volume:**
| File | Format | Description |
| --- | --- | --- |
| `bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026.zip` | GeoPackage (.gpkg) inside zip | FCC BDC 5G NR H3 coverage for WA state (FIPS 53) |
| `Washington.zip` | Shapefile (.shp + sidecar files) inside zip | Microsoft building footprints for Washington |
| `310.csv.gz` | Gzipped CSV, no header | OpenCellID tower data for MCC 310 (USA) |

**Target delta tables:**
| Table | Key Columns | Filter / Transform |
| --- | --- | --- |
| `fcc_bdc_h3_seattle` | fid, technology, mindown, minup, environmnt, h3_res9_id | Filter H3 hexagons to Seattle bbox (lat 47.40\~47.80, lon -122.49\~-122.10) |
| `building_footprints` | building_id, height, geometry (4326) | Read shapefile polygons into native Databricks geometry type |
| `cell_towers` | tower_id, carrier, tower_type, lac_tac, cell_id, coverage_radius_m, samples, latitude, longitude, created_ts, updated_ts, avg_signal, location (4326) | Filter to T-Mobile (MNC 260) + Seattle bbox; add POINT geometry |

**Seattle bounding box:** lat ∈ [47.40, 47.80], lon ∈ [-122.49, -122.10]

In [0]:
%pip install geopandas --quiet

In [0]:
# ---- Parameterized config ----
CATALOG = "cmegdemos_catalog"
SCHEMA  = "network_enablement_demo_test"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"

# Seattle bounding box (greater metro area)
SEATTLE_LAT_MIN, SEATTLE_LAT_MAX = 47.40, 47.80
SEATTLE_LON_MIN, SEATTLE_LON_MAX = -122.49, -122.10

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA}")
print(f"Volume  : {VOLUME_PATH}")
print(f"Seattle bbox: lat [{SEATTLE_LAT_MIN}, {SEATTLE_LAT_MAX}], lon [{SEATTLE_LON_MIN}, {SEATTLE_LON_MAX}]")

In [0]:
import zipfile, tempfile, os, sqlite3
import pandas as pd

# ---------- Extract GeoPackage from zip ----------
bdc_zip = f"{VOLUME_PATH}/bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026.zip"
tmp_dir = tempfile.mkdtemp()
with zipfile.ZipFile(bdc_zip, 'r') as z:
    z.extractall(tmp_dir)

gpkg_path = os.path.join(tmp_dir, "bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026.gpkg")
table_name = "bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026"

# ---------- Read from GeoPackage (SQLite) with rtree spatial pre-filter ----------
conn = sqlite3.connect(gpkg_path)
query = f"""
SELECT t.fid, t.technology, t.mindown, t.minup, t.environmnt, t.h3_res9_id
FROM "{table_name}" t
WHERE t.fid IN (
    SELECT id FROM "rtree_{table_name}_geom"
    WHERE maxx >= {SEATTLE_LON_MIN} AND minx <= {SEATTLE_LON_MAX}
      AND maxy >= {SEATTLE_LAT_MIN} AND miny <= {SEATTLE_LAT_MAX}
)
"""
pdf_bdc = pd.read_sql_query(query, conn)
conn.close()
print(f"Rows after rtree spatial pre-filter: {len(pdf_bdc)}")

# ---------- Load into Spark and refine filter using H3 center coords ----------
sdf_bdc = spark.createDataFrame(pdf_bdc)
sdf_bdc.createOrReplaceTempView("bdc_raw")

spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.fcc_bdc_h3_seattle AS
SELECT fid, 
       CAST(technology AS BIGINT) AS technology, 
       CAST(mindown AS DOUBLE) AS mindown, 
       CAST(minup AS DOUBLE) AS minup, 
       CAST(environmnt AS BIGINT) AS environmnt, 
       h3_res9_id
FROM bdc_raw
WHERE ST_Y(ST_GeomFromWKT(h3_centeraswkt(h3_res9_id))) BETWEEN {SEATTLE_LAT_MIN} AND {SEATTLE_LAT_MAX}
  AND ST_X(ST_GeomFromWKT(h3_centeraswkt(h3_res9_id))) BETWEEN {SEATTLE_LON_MIN} AND {SEATTLE_LON_MAX}
""")

count = spark.sql(f"SELECT count(*) as cnt FROM {CATALOG}.{SCHEMA}.fcc_bdc_h3_seattle").collect()[0][0]
print(f"✓ fcc_bdc_h3_seattle: {count:,} rows")
display(spark.table(f"{CATALOG}.{SCHEMA}.fcc_bdc_h3_seattle").limit(5))

In [0]:
import geopandas as gpd
import zipfile, tempfile, os

# ---------- Extract shapefile from zip ----------
bldg_zip = f"{VOLUME_PATH}/Washington.zip"
tmp_dir = tempfile.mkdtemp()
with zipfile.ZipFile(bldg_zip, 'r') as z:
    z.extractall(tmp_dir)

# ---------- Read shapefile with geopandas ----------
gdf = gpd.read_file(os.path.join(tmp_dir, "bldg_footprints.shp"))
print(f"Total footprints in shapefile: {len(gdf):,}")
print(f"Columns: {list(gdf.columns)}")
print(f"CRS: {gdf.crs}")
print(gdf.head(3))

In [0]:
from shapely.geometry import box

# ---------- Filter to Seattle bounding box ----------
seattle_bbox = box(SEATTLE_LON_MIN, SEATTLE_LAT_MIN, SEATTLE_LON_MAX, SEATTLE_LAT_MAX)
gdf_seattle = gdf[gdf.geometry.intersects(seattle_bbox)].copy()
print(f"Footprints in Seattle area: {len(gdf_seattle):,}")

# ---------- Convert geometry to WKT for Spark ----------
gdf_seattle['geometry_wkt'] = gdf_seattle.geometry.to_wkt()
pdf_bldg = gdf_seattle[['Height', 'geometry_wkt']]

# ---------- Create Spark DataFrame and save with native geometry (SRID 4326) ----------
sdf_bldg = spark.createDataFrame(pdf_bldg)
sdf_bldg.createOrReplaceTempView("bldg_raw")

spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.building_footprints AS
SELECT 
  monotonically_increasing_id() AS building_id,
  CAST(Height AS DOUBLE) AS height,
  ST_SetSRID(ST_GeomFromWKT(geometry_wkt), 4326) AS geometry
FROM bldg_raw
""")

count = spark.sql(f"SELECT count(*) as cnt FROM {CATALOG}.{SCHEMA}.building_footprints").collect()[0][0]
print(f"✓ building_footprints: {count:,} rows")
display(spark.table(f"{CATALOG}.{SCHEMA}.building_footprints").limit(5))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType

# ---------- Define schema (OpenCellID CSV has no header) ----------
ocid_schema = StructType([
    StructField("radio", StringType()),
    StructField("mcc", IntegerType()),
    StructField("net", IntegerType()),
    StructField("area", IntegerType()),
    StructField("cell", LongType()),
    StructField("unit", IntegerType()),
    StructField("lon", DoubleType()),
    StructField("lat", DoubleType()),
    StructField("range", IntegerType()),
    StructField("samples", IntegerType()),
    StructField("changeable", IntegerType()),
    StructField("created", LongType()),
    StructField("updated", LongType()),
    StructField("averageSignal", IntegerType()),
])

# ---------- Read, filter to T-Mobile + Seattle, add geometry ----------
sdf_towers = (
    spark.read.csv(f"{VOLUME_PATH}/310.csv.gz", schema=ocid_schema, header=False)
    .filter(
        (F.col("net") == 260) &  # T-Mobile MNC
        (F.col("lat").between(SEATTLE_LAT_MIN, SEATTLE_LAT_MAX)) &
        (F.col("lon").between(SEATTLE_LON_MIN, SEATTLE_LON_MAX))
    )
)
sdf_towers.createOrReplaceTempView("towers_raw")
print(f"T-Mobile Seattle towers: {sdf_towers.count():,}")

# ---------- Save with proper column names, timestamps, and POINT geometry (SRID 4326) ----------
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.cell_towers AS
SELECT 
  monotonically_increasing_id() AS tower_id,
  'T-Mobile' AS carrier,
  radio AS tower_type,
  area AS lac_tac,
  cell AS cell_id,
  `range` AS coverage_radius_m,
  samples,
  lat AS latitude,
  lon AS longitude,
  CAST(from_unixtime(created) AS STRING) AS created_ts,
  CAST(from_unixtime(updated) AS STRING) AS updated_ts,
  averageSignal AS avg_signal,
  ST_SetSRID(ST_Point(lon, lat), 4326) AS location
FROM towers_raw
""")

count = spark.sql(f"SELECT count(*) as cnt FROM {CATALOG}.{SCHEMA}.cell_towers").collect()[0][0]
print(f"✓ cell_towers: {count:,} rows")
display(spark.table(f"{CATALOG}.{SCHEMA}.cell_towers").limit(5))

In [0]:
# ---------- Validate all three tables ----------
tables = ["fcc_bdc_h3_seattle", "building_footprints", "cell_towers"]

for t in tables:
    fqn = f"{CATALOG}.{SCHEMA}.{t}"
    count = spark.sql(f"SELECT count(*) FROM {fqn}").collect()[0][0]
    cols = [c.name for c in spark.table(fqn).schema]
    print(f"\n{'='*60}")
    print(f"{fqn}: {count:,} rows")
    print(f"Columns: {cols}")

# ---------- Quick H3 / ST function smoke tests ----------
print("\n" + "="*60)
print("H3 function test on fcc_bdc_h3_seattle:")
display(spark.sql(f"""
  SELECT h3_res9_id, 
         h3_centeraswkt(h3_res9_id) AS center_wkt,
         h3_boundaryaswkt(h3_res9_id) AS hex_boundary_wkt
  FROM {CATALOG}.{SCHEMA}.fcc_bdc_h3_seattle LIMIT 3
"""))

print("ST function test on building_footprints:")
display(spark.sql(f"""
  SELECT building_id, height, 
         ST_Area(geometry) AS area_sq_deg,
         ST_Centroid(geometry) AS centroid
  FROM {CATALOG}.{SCHEMA}.building_footprints LIMIT 3
"""))

print("ST function test on cell_towers:")
display(spark.sql(f"""
  SELECT tower_id, carrier, tower_type, 
         ST_X(location) AS lon, 
         ST_Y(location) AS lat
  FROM {CATALOG}.{SCHEMA}.cell_towers LIMIT 3
"""))